In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Core imports used across the notebook
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
target_col = 'Delivery_Time'
X = df.drop(columns=[target_col], axis=1)
y = df[target_col]
print(X.shape, y.shape)

In [ ]:
# Task 5: Write your code here:
print(y.value_counts())
y.value_counts().plot(kind='hist')
plt.title('Target counts')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1)

In [ ]:
df.head()

In [ ]:
# Task 2: Write your code here:
#remove rows where target is missing
df = df.dropna(subset='Delivery_Time')
df.head()

In [ ]:
(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
df.head()

In [ ]:

cols = ['Distance_km', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Weather']
for col in cols:
  df[col] = df[col].fillna(df[col].mode()[0])



In [ ]:
(df.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
df.head()

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns


for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
target_col = 'Delivery_Time'
X = df.drop(columns=[target_col], axis=1)
y = df[target_col]
print(X.shape, y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
# Task 5: Write your code here:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [ ]:
# Task 2,3,4,5: Write your code here:


# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)


print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = df.drop(columns=[target_col], axis=1).columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
################################## PLOT ################################
y_pred = model.predict(X_test)
y_pred.plot(kind='hist')
# Task 5: Write your code here:
# print(y_pred.value_counts())
# y_pred.value_counts().plot(kind='hist')
plt.title('Target counts')
plt.show()

In [ ]:
# Task Bonus: Write your code here: